In [ ]:
import os
import re
import collections
from osgeo import gdal
from tqdm.notebook import tqdm

from typing import List

gdal.UseExceptions()

In [ ]:
def group_geotiffs_by_basename(input_folder: str):
    """
    Groups GeoTIFFs by a common basename from GEE export format.

    Args:
        input_folder (str): Folder containing the exported GeoTIFF shards. Files
            that do not match the GEE export naming pattern are skipped.

    Returns:
        defaultdict[str, List[str]]: Basename mapped to the full paths of its
            shards.
    """
    grouped_files = collections.defaultdict(list)
    
    # Use a non-greedy match (.*?) to correctly capture the basename
    pattern = re.compile(r'^(.*?)-/d{10}-/d{10}/.tif$')

    for filename in os.listdir(input_folder):
        if filename.lower().endswith('.tif'):
            match = pattern.match(filename)
            if match:
                basename = match.group(1)
                full_path = os.path.join(input_folder, filename)
                grouped_files[basename].append(full_path)
            else:
                print(f"  - Skipping file with non-matching name: {filename}")

    return grouped_files


In [ ]:
def create_mosaic(
    basename: str, 
    file_list: List[str], 
    output_folder: str
    ):
    """
    Creates a single mosaicked GeoTIFF from a list of input GeoTIFFs.

    Args:
        basename (str): Name stem used for the temporary VRT and the output
            mosaic file.
        file_list (List[str]): Paths to the input GeoTIFFs to mosaic. The first
            file supplies the NoData value, data type, and band names.
        output_folder (str): Folder where the mosaic is written.

    Returns:
        None
    """
    print(f"/n--- Processing group: {basename} ---")
    print(f"  - Mosaicking {len(file_list)} files.")

    vrt_path = os.path.join(output_folder, f"{basename}_temp.vrt")
    output_mosaic_path = os.path.join(output_folder, f"{basename}_mosaic.tif")

    try:
        source_ds = gdal.Open(file_list[0], gdal.GA_ReadOnly)
        if source_ds is None:
            print(f"  - ERROR: Could not open {file_list[0]} to read metadata.")
            return

        source_band = source_ds.GetRasterBand(1)
        nodata_value = source_band.GetNoDataValue()
        data_type = source_band.DataType
        band_names = [source_ds.GetRasterBand(i + 1).GetDescription() for i in range(source_ds.RasterCount)]
        source_ds = None

        print(f"  - Source NoData Value: {nodata_value}")
        print(f"  - Source Data Type: {gdal.GetDataTypeName(data_type)}")

    except Exception as e:
        print(f"  - ERROR: Failed to read metadata from source files. {e}")
        return

    print(f"  - Building VRT: {os.path.basename(vrt_path)}")
    vrt_options = gdal.BuildVRTOptions(resampleAlg='nearest', addAlpha=False, separate=False)
    gdal.BuildVRT(vrt_path, file_list, options=vrt_options)

    print(f"  - Warping VRT to create final mosaic: {os.path.basename(output_mosaic_path)}")
    warp_options = gdal.WarpOptions(
        format = 'GTiff',
        dstSRS = 'EPSG:5070',
        xRes = 30,
        yRes = 30,
        resampleAlg = 'nearest',
        dstNodata = nodata_value,
        outputType = data_type,
        creationOptions = ['COMPRESS=LZW', 'TILED=YES', 'BIGTIFF=YES']
    )
    
    try:
        
        gdal.Warp(output_mosaic_path, vrt_path, options=warp_options)

        # Set band names on the output file, as they are not always preserved by Warp.
        dest_ds = gdal.Open(output_mosaic_path, gdal.GA_Update)
        if dest_ds:
            for i, name in enumerate(band_names):
                if name:
                    band = dest_ds.GetRasterBand(i + 1)
                    band.SetDescription(name)
            dest_ds.FlushCache()
            dest_ds = None
        
        print(f"  - Mosaic created successfully: {output_mosaic_path}")

    except Exception as e:
        print(f"  - ERROR: An error occurred during the warp operation: {e}")

    finally:
        if os.path.exists(vrt_path):
            os.remove(vrt_path)
            print(f"  - Cleaned up temporary VRT file.")

    return None
            

In [ ]:
def run_annual_mtbs(repo_folder: str):
    """
    Mosaic the annual MTBS severity shards into one raster per year.

    Args:
        repo_folder (str): Path to the repository root. Input and output folders
            are resolved relative to this path.

    Returns:
        None
    """
    input_folder = f"{repo_folder}/data/rasters/mtbs_annual_severity_shards"
    output_folder = f"{repo_folder}/data/rasters/mtbs_severity_annual"
    
    if not os.path.isdir(input_folder):
        print(f"Error: Input folder not found at '{input_folder}'")
    else:
        if not os.path.exists(output_folder):
            print(f"Output folder not found. Creating it at: {output_folder}")
            os.makedirs(output_folder)
            
        grouped_tiffs = group_geotiffs_by_basename(input_folder)
    
        if not grouped_tiffs:
            print("No files matching the expected pattern were found. Exiting.")
        else:
            for basename, file_list in tqdm(grouped_tiffs.items()):
                create_mosaic(basename, file_list, output_folder)
        
        print("/nAll processing complete.")

    return None

In [ ]:
def run_nlcd_forest_mask(repo_folder: str):
    """
    Mosaic the NLCD forest mask shards into complete rasters.

    Args:
        repo_folder (str): Path to the repository root. Input and output folders
            are resolved relative to this path.

    Returns:
        None
    """   
    input_folder = f"{repo_folder}/data/rasters/ncld_forest_mask_shards"
    output_folder = f"{repo_folder}/data/rasters/ncld_forest_mask"
    
    if not os.path.isdir(input_folder):
        print(f"Error: Input folder not found at '{input_folder}'")
    else:
        if not os.path.exists(output_folder):
            print(f"Output folder not found. Creating it at: {output_folder}")
            os.makedirs(output_folder)
            
        grouped_tiffs = group_geotiffs_by_basename(input_folder)
    
        if not grouped_tiffs:
            print("No files matching the expected pattern were found. Exiting.")
        else:
            for basename, file_list in tqdm(grouped_tiffs.items()):
                create_mosaic(basename, file_list, output_folder)
        
        print("/nAll processing complete.")

    return None

## File I/O

In [ ]:
# Define a path to the project folder
data_folder = "<PATH/TO/PROJECT/FOLDER>/data"

## Mosaic the annual MTBS rasters

In [ ]:
run_annual_mtbs(data_folder)

## Mosaic the NLCD forest/non-forest mask

In [ ]:
run_nlcd_forest_mask(data_folder)